# 01 Preprocessing

Preprocessing-only notebook for loading cleaned ASHRAE data and building engineered features.


## Scope
- Load cleaned train/weather/building metadata tables
- Merge and validate columns
- Build preprocessing features (time, weather, metadata, lag/rolling, encodings)


## 1. Environment Setup and Data Loading

In [1]:
import os, random, math, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean, cosine as cosine_dist
from scipy.signal import savgol_filter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## 1a. Generate Cleaned Input Files
Run these once (or whenever raw data/preprocessing params change) before loading data below.


In [2]:
import subprocess, sys

RUN_PREPROCESS_SCRIPTS = True  

if RUN_PREPROCESS_SCRIPTS:
    cmd = [sys.executable, "ashrae/preprocess_isamu_matt.py", "--input-dir", "ashrae", "--output-dir", "ashrae"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipped: preprocess_isamu_matt.py")


Running: c:\Users\tamar\AppData\Local\Programs\Python\Python314\python.exe ashrae/preprocess_isamu_matt.py --input-dir ashrae --output-dir ashrae


In [3]:
import subprocess, sys

RUN_MEAN_FILTER_SCRIPT = True  

if RUN_MEAN_FILTER_SCRIPT:
    cmd = [sys.executable, "ashrae/make_building_mean_y_ge_1.py"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipped: make_building_mean_y_ge_1.py")


Running: c:\Users\tamar\AppData\Local\Programs\Python\Python314\python.exe ashrae/make_building_mean_y_ge_1.py


In [4]:
# TRAIN_PATH = "ashrae/train.cleaned_isamu_matt.building_mean_y_ge_1.csv"
# BMETA_PATH = "ashrae/building_metadata.csv"
# WTRAIN_PATH = "ashrae/weather_train.cleaned.csv"

# MAX_BUILDINGS_LOAD = None
# CHUNK_SIZE = 500_000

# bmeta = pd.read_csv(BMETA_PATH)
# wtrain = pd.read_csv(
#     WTRAIN_PATH,
#     dtype={
#         "site_id": "int8",
#         "air_temperature": "float32",
#         "cloud_coverage": "float32",
#         "dew_temperature": "float32",
#         "precip_depth_1_hr": "float32",
#         "sea_level_pressure": "float32",
#         "wind_direction": "float32",
#         "wind_speed": "float32",
#     },
# )
# wtrain["timestamp"] = pd.to_datetime(wtrain["timestamp"])

# counts = {}
# for ch in pd.read_csv(
#     TRAIN_PATH,
#     usecols=["building_id", "meter"],
#     dtype={"building_id": "int16", "meter": "int8"},
#     chunksize=CHUNK_SIZE,
# ):
#     ch = ch[ch["meter"] == 0]
#     vc = ch["building_id"].value_counts()
#     for bid, c in vc.items():
#         counts[int(bid)] = counts.get(int(bid), 0) + int(c)

# all_bids = sorted(counts.keys())
# if MAX_BUILDINGS_LOAD is None or MAX_BUILDINGS_LOAD >= len(all_bids):
#     chosen_bids = set(all_bids)
# else:
#     rng = np.random.default_rng(SEED)
#     chosen_bids = set(rng.choice(np.array(all_bids), size=MAX_BUILDINGS_LOAD, replace=False).tolist())
# parts = []
# for ch in pd.read_csv(
#     TRAIN_PATH,
#     usecols=["building_id", "meter", "timestamp", "meter_reading", "site_id", "square_feet"],
#     dtype={
#         "building_id": "int16",
#         "meter": "int8",
#         "meter_reading": "float32",
#         "site_id": "int8",
#         "square_feet": "float32",
#     },
#     chunksize=CHUNK_SIZE,
# ):
#     ch = ch[(ch["meter"] == 0) & (ch["building_id"].isin(chosen_bids))]
#     if not ch.empty:
#         parts.append(ch)

# train = pd.concat(parts, ignore_index=True)
# train["timestamp"] = pd.to_datetime(train["timestamp"])

# extra_cols = [c for c in bmeta.columns if c not in train.columns]
# if extra_cols:
#     train = train.merge(bmeta[["building_id"] + extra_cols], on="building_id", how="left", copy=False)

# df = train.merge(wtrain, on=["site_id", "timestamp"], how="left", copy=False)

# df.sort_values(["building_id", "timestamp"], inplace=True, kind="mergesort")

# print(f"Loaded buildings: {train['building_id'].nunique()} / {len(all_bids)}")
# print("Merged shape (preprocessed):", df.shape)
# df.head()


In [5]:
print("train cols:", train.columns.tolist())
print("bmeta cols:", bmeta.columns.tolist())
print("wtrain cols:", wtrain.columns.tolist())

print("\ntrain head:")
display(train.head(2))

print("\nbmeta head:")
display(bmeta.head(2))

print("\nwtrain head:")
display(wtrain.head(2))

print("\nsite_id in train?", "site_id" in train.columns)
print("site_id in bmeta?", "site_id" in bmeta.columns)
print("site_id in wtrain?", "site_id" in wtrain.columns)

site_like_train = [x for x in train.columns if "site_id" in x]
site_like_bmeta = [x for x in bmeta.columns if "site_id" in x]
print("\nsite-like columns in train:", site_like_train)
print("site-like columns in bmeta:", site_like_bmeta)


NameError: name 'train' is not defined

## 2. Feature Engineering

In [ ]:
# ================================================================
# Feature Engineering (following Kaggle 1st-place solution patterns)
# ================================================================


df["hour"] = df["timestamp"].dt.hour
df["dow"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["is_weekend"] = (df["dow"] >= 5).astype(int)


df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"]  = np.sin(2 * np.pi * df["dow"] / 7)
df["dow_cos"]  = np.cos(2 * np.pi * df["dow"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)


df["log_sqft"] = np.log1p(df["square_feet"].fillna(df["square_feet"].median()))
df["year_built"] = df["year_built"].fillna(df["year_built"].median())
df["building_age"] = 2017 - df["year_built"]  
df["floor_count"] = df["floor_count"].fillna(1)


weather_cols = [
    "air_temperature", "dew_temperature", "wind_speed", "cloud_coverage",
    "precip_depth_1_hr", "sea_level_pressure"
]
for c in weather_cols:
    if c in df.columns:
        df[c] = df[c].fillna(df[c].median())


def apply_savgol_features(group):
    """Apply Savitzky-Golay filter for smoothed temperature and derivatives"""
    temp = group["air_temperature"].values
    n = len(temp)
    if n > 25:
        window = min(25, n // 2 * 2 + 1)  
        if window % 2 == 0:
            window -= 1
        if window >= 5:
            group = group.copy()
            group["temp_smooth"] = savgol_filter(temp, window, 3)
            group["temp_diff1"] = savgol_filter(temp, window, 3, deriv=1)
        else:
            group = group.copy()
            group["temp_smooth"] = temp
            group["temp_diff1"] = 0.0
    else:
        group = group.copy()
        group["temp_smooth"] = temp
        group["temp_diff1"] = 0.0
    return group

print("Applying Savitzky-Golay temperature features...")

_building_ids = df["building_id"].values
df = df.groupby("building_id", group_keys=False).apply(apply_savgol_features)
df = df.reset_index(drop=True)
if "building_id" not in df.columns:
    df["building_id"] = _building_ids
df["temp_smooth"] = df["temp_smooth"].fillna(df["air_temperature"])
df["temp_diff1"] = df["temp_diff1"].fillna(0)

df["temp_lag_1"]  = df.groupby("building_id")["air_temperature"].shift(1)
df["temp_lag_3"]  = df.groupby("building_id")["air_temperature"].shift(3)
df["temp_lag_24"] = df.groupby("building_id")["air_temperature"].shift(24)
df["temp_lag_1"]  = df["temp_lag_1"].fillna(df["air_temperature"])
df["temp_lag_3"]  = df["temp_lag_3"].fillna(df["air_temperature"])
df["temp_lag_24"] = df["temp_lag_24"].fillna(df["air_temperature"])

for lag in [1, 24]:
    df[f"lag_{lag}"] = df.groupby("building_id")["meter_reading"].shift(lag)

df["roll24_mean"] = df.groupby("building_id")["meter_reading"].shift(1).rolling(24).mean()
df["roll24_std"]  = df.groupby("building_id")["meter_reading"].shift(1).rolling(24).std()

for _c in ['lag_1', 'lag_24', 'roll24_mean', 'roll24_std']:
    df[_c] = df[_c].astype('float32')
valid_rows = df[['lag_1', 'lag_24', 'roll24_mean', 'roll24_std']].notna().all(axis=1).to_numpy()
df = df.loc[valid_rows].copy()
df.reset_index(drop=True, inplace=True)

primary_use_cats = sorted(df["primary_use"].dropna().unique())
PRIMARY_USE_MAP = {v: i for i, v in enumerate(primary_use_cats)}
df["primary_use_enc"] = df["primary_use"].map(PRIMARY_USE_MAP).fillna(0).astype(int)

df["site_id_enc"] = df["site_id"].astype(int)

df["y"] = np.log1p(df["meter_reading"].values)

# ================================================================
# Define feature sets
# ================================================================

NUM_FEATURES = [
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "is_weekend",
    "log_sqft", "building_age", "floor_count",
    "air_temperature", "dew_temperature", "wind_speed", "cloud_coverage",
    "precip_depth_1_hr", "sea_level_pressure",
    "temp_smooth", "temp_diff1", "temp_lag_1", "temp_lag_3", "temp_lag_24",
    "lag_1", "lag_24", "roll24_mean", "roll24_std",
]


CAT_FEATURES = ["site_id_enc", "primary_use_enc"]


CAT_CONFIGS = [
    (16 + 1, 4),                           # site_id: 16 sites → 4-dim embedding
    (len(PRIMARY_USE_MAP) + 1, 4),         # primary_use: ~16 types → 4-dim embedding
]


FEATURES = NUM_FEATURES + CAT_FEATURES
N_NUM = len(NUM_FEATURES)
N_CAT = len(CAT_FEATURES)
CAT_INDICES = list(range(N_NUM, N_NUM + N_CAT))

TARGET = "y"

print(f"Numerical features: {N_NUM}")
print(f"Categorical features: {N_CAT} (with embeddings)")
print(f"Total feature columns: {len(FEATURES)}")
print(f"Embedding configs: {CAT_CONFIGS}")
df[FEATURES + ["building_id", "site_id", "timestamp", "meter_reading", "y"]].head()

Applying Savitzky-Golay temperature features...


C:\Users\tamar\AppData\Local\Temp\ipykernel_15968\1085266870.py:60: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("building_id", group_keys=False).apply(apply_savgol_features)


Numerical features: 25
Categorical features: 2 (with embeddings)
Total feature columns: 27
Embedding configs: [(17, 4), (17, 4)]


,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,is_weekend,log_sqft,building_age,floor_count,...,lag_24,roll24_mean,roll24_std,site_id_enc,primary_use_enc,building_id,site_id,timestamp,meter_reading,y
0,0.000000,1.000000,-0.974928,-0.222521,0.5,0.866025,1,8.913685,9.0,1.0,...,0.0,0.0,0.0,0,0,0,0,2016-01-02 00:00:00,0.0,0.0
1,0.258819,0.965926,-0.974928,-0.222521,0.5,0.866025,1,8.913685,9.0,1.0,...,0.0,0.0,0.0,0,0,0,0,2016-01-02 01:00:00,0.0,0.0
2,0.500000,0.866025,-0.974928,-0.222521,0.5,0.866025,1,8.913685,9.0,1.0,...,0.0,0.0,0.0,0,0,0,0,2016-01-02 02:00:00,0.0,0.0
3,0.707107,0.707107,-0.974928,-0.222521,0.5,0.866025,1,8.913685,9.0,1.0,...,0.0,0.0,0.0,0,0,0,0,2016-01-02 03:00:00,0.0,0.0
4,0.866025,0.500000,-0.974928,-0.222521,0.5,0.866025,1,8.913685,9.0,1.0,...,0.0,0.0,0.0,0,0,0,0,2016-01-02 04:00:00,0.0,0.0


## Result
df contains the preprocessed dataset ready for downstream training.
